# Headline

## Download

In [2]:
!pip install vnstock
!pip install pmdarima
!pip install arch
!pip install nolds

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.5/277.5 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.7/225.7 kB 11.2 MB/s eta 0:00:00


## Import

In [3]:
from vnstock import *
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import pmdarima as pm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

from arch import arch_model
from statsmodels.stats.diagnostic import het_arch

import nolds
from statsmodels.tsa.stattools import bds

# Data Input

In [4]:
quote = Quote(symbol='DIG', source='KBS')
raw_historical_data = quote.history(start='2020-01-01', end='2025-05-25')

data = raw_historical_data.drop(['time'], axis=1)


📋 Connecting Google Drive account
to save project settings.

Mounted at /content/drive


# Data Preparation:

In [5]:
data["returns"] = data.close.diff()
data["log_return"] = np.log(data.close).diff()

data.dropna(inplace=True)
data.head()

,open,high,low,close,volume,returns,log_return
1,7.63,7.82,7.63,7.68,2242240,0.05,0.006532
2,7.71,7.71,7.54,7.57,470820,-0.11,-0.014426
3,7.51,7.68,7.51,7.54,424110,-0.03,-0.003971
4,7.49,7.49,7.16,7.16,2280630,-0.38,-0.051712
5,7.24,7.38,7.24,7.27,553820,0.11,0.015246


# Test for mean predictability

## Stationarity Test (ADF):

In [6]:
def check_stationarity(data, name):
    result = adfuller(data)
    print(f"--- ADF Test: {name} ---")
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    if result[1] < 0.05:
        print("Result: Stationary (Reject Null Hypothesis)")
    else:
        print("Result: Non-Stationary (Fail to Reject Null)")
    print("")

check_stationarity(data.close, "Price")
check_stationarity(data.returns, "Return")
check_stationarity(data.log_return, "Log Return")

--- ADF Test: Price ---
ADF Statistic: -1.7189
p-value: 0.4214
Result: Non-Stationary (Fail to Reject Null)

--- ADF Test: Return ---
ADF Statistic: -7.9055
p-value: 0.0000
Result: Stationary (Reject Null Hypothesis)

--- ADF Test: Log Return ---
ADF Statistic: -8.8022
p-value: 0.0000
Result: Stationary (Reject Null Hypothesis)



taking $d=1$

## ARIMA model

In [7]:
# Let Python find the best parameters for the MEAN model
# This effectively "fixes" the Mean equation before you add GARCH
best_model = pm.auto_arima(data['log_return'].dropna(),
                           start_p=1, start_q=1, d=0,
                           max_p=8, max_q=8, # Search wider
                           vol='Garch', dist='t',
                           seasonal=False,
                           stepwise=True,
                           trace=True)

print(best_model.summary())

Performing stepwise search to minimize aic
 ARIMA(1,0,1)(0,0,0)[0]             : AIC=-5307.056, Time=2.41 sec
 ARIMA(0,0,0)(0,0,0)[0]             : AIC=-5286.572, Time=0.62 sec
 ARIMA(1,0,0)(0,0,0)[0]             : AIC=-5305.752, Time=0.92 sec
 ARIMA(0,0,1)(0,0,0)[0]             : AIC=-5304.255, Time=2.63 sec
 ARIMA(2,0,1)(0,0,0)[0]             : AIC=-5309.212, Time=1.95 sec
 ARIMA(2,0,0)(0,0,0)[0]             : AIC=-5305.371, Time=0.23 sec
 ARIMA(3,0,1)(0,0,0)[0]             : AIC=-5303.220, Time=0.37 sec
 ARIMA(2,0,2)(0,0,0)[0]             : AIC=-5307.346, Time=0.35 sec
 ARIMA(1,0,2)(0,0,0)[0]             : AIC=-5309.254, Time=0.51 sec
 ARIMA(0,0,2)(0,0,0)[0]             : AIC=-5304.336, Time=0.26 sec
 ARIMA(1,0,3)(0,0,0)[0]             : AIC=-5305.994, Time=0.37 sec
 ARIMA(0,0,3)(0,0,0)[0]             : AIC=-5304.213, Time=0.15 sec
 ARIMA(2,0,3)(0,0,0)[0]             : AIC=-5305.642, Time=1.34 sec
 ARIMA(1,0,2)(0,0,0)[0] intercept   : AIC=-5307.362, Time=1.12 sec

Best model:  ARIMA

## Ljung–Box (Autocorrelation test):

In [13]:
def check_autocorrelation(data, name, lags=[5, 10, 15, 20]):
    # returns a dataframe in newer statsmodels versions
    lb_df = acorr_ljungbox(data, lags=lags, return_df=True)

    print(lb_df)

    p_val = lb_df['lb_pvalue'].iloc[0]
    print(f"--- Ljung-Box Test: {name} ---")
    print(f"p-value (at lag {lags}): {p_val:.4f}")
    if p_val < 0.05:
        print("Result: Autocorrelation Detected")
    else:
        print("Result: No Significant Autocorrelation")
    print("")

check_autocorrelation(best_model.resid(), "Residuals")

      lb_stat  lb_pvalue
5    0.108403   0.999802
10  15.975074   0.100348
15  33.870134   0.003551
20  48.874660   0.000320
--- Ljung-Box Test: Residuals ---
p-value (at lag [5, 10, 15, 20]): 0.9998
Result: No Significant Autocorrelation



# Test for volatility predictability

## ARCH effect test:

In [9]:
# het_arch returns: (LM-statistic, p-value, F-statistic, p-value for F-statistic)
arch_test_results = het_arch(best_model.resid())

lm_statistic, p_value_lm, f_statistic, p_value_f = arch_test_results

print("--- ARCH Effect Test (Lagrange Multiplier Test) ---")
print(f"LM Statistic: {lm_statistic:.4f}")
print(f"p-value (LM): {p_value_lm:.4f}")
print(f"F Statistic: {f_statistic:.4f}")
print(f"p-value (F-statistic): {p_value_f:.4f}")

if p_value_lm < 0.05:
    print("Result: ARCH effects detected (Reject Null Hypothesis of no ARCH effects)")
else:
    print("Result: No significant ARCH effects (Fail to Reject Null Hypothesis)")
print("")

--- ARCH Effect Test (Lagrange Multiplier Test) ---
LM Statistic: 310.8378
p-value (LM): 0.0000
F Statistic: 40.2107
p-value (F-statistic): 0.0000
Result: ARCH effects detected (Reject Null Hypothesis of no ARCH effects)



## GARCH model

In [10]:
# The best_model from pmdarima was ARIMA(1,0,2) on log_return
# This means for the log_return, we have an ARMA(1,2) mean equation.

# Fit an ARMA(1,2)-GARCH(1,1) model
# mean='ARMA' for combined AR and MA terms in the mean equation
# p=1 for the AR order in the mean equation
# q=2 for the MA order in the mean equation
# vol='Garch' for GARCH variance model
# p=1, q=1 for GARCH(1,1) in the variance equation
# dist='t' for Student's t-distribution for errors, often better for financial data

gm = arch_model(best_model.resid(),
                mean='Zero',
                vol='Garch',
                p=1, # ARCH order for variance
                q=1, # GARCH order for variance
                dist='t')

# Fit the model
# Removing disp='off' to see the summary output
res = gm.fit(update_freq=5)

# Print the summary of the GARCH model
print(res.summary())

Optimization terminated successfully    (Exit mode 0)
            Current function value: -2806.4486096005203
            Iterations: 5
            Function evaluations: 5
            Gradient evaluations: 1
                          Zero Mean - GARCH Model Results                           
Dep. Variable:                         None   R-squared:                       0.000
Mean Model:                       Zero Mean   Adj. R-squared:                  0.001
Vol Model:                            GARCH   Log-Likelihood:                2806.45
Distribution:      Standardized Student's t   AIC:                          -5604.90
Method:                  Maximum Likelihood   BIC:                          -5584.09
                                              No. Observations:                 1342
Date:                      Fri, Feb 06 2026   Df Residuals:                     1342
Time:                              15:18:15   Df Model:                            0
                           

y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.001114. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.



## Ljung-Box test:

In [14]:
# 1. Get the standardized residuals from your NEW 'Zero Mean' model
# (Make sure 'res' is the variable name of the result you just pasted)
final_std_resid = res.resid / res.conditional_volatility

# 2. Test Squared Residuals (Checks if Volatility Clustering is removed)
lb_test = acorr_ljungbox(final_std_resid, lags=[10, 15, 20], return_df=True)

print("--- FINAL DIAGNOSIS (Squared Residuals) ---")
print(lb_test)

--- FINAL DIAGNOSIS (Squared Residuals) ---
      lb_stat  lb_pvalue
10  10.044326   0.436613
15  17.248817   0.304209
20  24.342975   0.227731


# Nonlinear predictability check


In [12]:
# Perform the BDS test on the standardized residuals
bds_statistic, bds_pvalue = bds(final_std_resid)

print("--- BDS Test for Non-linear Predictability (Standardized Residuals) ---")
print(f"BDS Statistic: {bds_statistic:.4f}")
print(f"p-value: {bds_pvalue:.4f}")

if bds_pvalue < 0.05:
    print("Result: Non-linear dependencies detected (Reject Null Hypothesis of i.i.d.)")
else:
    print("Result: No significant non-linear dependencies detected (Fail to Reject Null Hypothesis of i.i.d.)")

--- BDS Test for Non-linear Predictability (Standardized Residuals) ---
BDS Statistic: 0.6506
p-value: 0.5153
Result: No significant non-linear dependencies detected (Fail to Reject Null Hypothesis of i.i.d.)
